# 00 - Proje Kurulum Kontrolü

Bu notebook, projenin temel klasör yapısını ve ham veri dizinlerinin beklenen düzende olup olmadığını kontrol eder.

Amaç model eğitimi yapmak değil; sonraki notebookların çalışması için gerekli başlangıç koşullarını doğrulamaktır.

## 1. Kütüphaneler

Bu bölümde dosya sistemi kontrolleri, tablo işlemleri ve basit görselleştirmeler için kullanılan kütüphaneler yüklenir.

In [ ]:
from pathlib import Path
from datetime import datetime
import warnings

import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

## 2. Ayarlar ve Dosya Yolları

Bu bölümde proje kök dizini, traditional ML yaklaşım klasörü ve bu notebooka ait input-output yolları tanımlanır.

Notebook çıktıları yalnızca kendi `tables` ve `figures` klasörlerine yazılır.

In [ ]:
def find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()

    start_path = Path(start_path).resolve()

    for candidate in [start_path] + list(start_path.parents):
        has_repo_layout = (candidate / "approaches" / "traditional_ml").exists()
        has_data_dir = (candidate / "data").exists()

        if has_repo_layout and has_data_dir:
            return candidate

    return None


PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    raise FileNotFoundError("Proje kökü bulunamadı. Notebook'u repo içinde çalıştırın.")

APPROACH_NAME = "traditional_ml"
STAGE_NAME = "00_project_setup"
NOTEBOOK_NAME = "00_project_setup"

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

DOCS_DIR = PROJECT_ROOT / "docs"
APPROACH_DIR = PROJECT_ROOT / "approaches" / APPROACH_NAME

NOTEBOOK_DIR = APPROACH_DIR / "notebooks" / STAGE_NAME
OUTPUT_DIR = APPROACH_DIR / "outputs" / STAGE_NAME / NOTEBOOK_NAME

TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

OVERWRITE_TABLES = False
OVERWRITE_FIGURES = False

RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

for directory in [OUTPUT_DIR, TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

## 3. Beklenen Proje Yapısı

Bu bölümde projede bulunması beklenen temel klasörler, yaklaşım klasörleri ve ham veri dizinleri tanımlanır.

In [ ]:
EXPECTED_TOP_LEVEL_FOLDERS = [
    "data",
    "docs",
    "approaches",
    "figures",
    "requirements",
]

EXPECTED_REPOSITORY_FILES = [
    "README.md",
    ".gitignore",
]

EXPECTED_APPROACH_FOLDERS = [
    "notebooks",
    "inputs",
    "outputs",
]

EXPECTED_DATASETS = {
    "dataset1_varroa": {
        "role": "model_development_internal",
        "splits": ["train", "valid", "test"],
    },
    "dataset2_honeybee": {
        "role": "model_development_internal",
        "splits": ["train", "valid", "test"],
    },
    "dataset3_external": {
        "role": "external_validation_only",
        "splits": ["train", "valid", "test"],
    },
}

EXPECTED_SPLIT_SUBFOLDERS = ["images", "labels"]

EXPECTED_DATASET_LEVEL_FILES = [
    "data.yaml",
    "README.dataset.txt",
    "README.roboflow.txt",
]

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LABEL_EXTENSIONS = {".txt"}

## 4. Yardımcı Fonksiyonlar

Bu bölümde kısa log mesajları yazdırmak, klasör-dosya varlığını kontrol etmek, CSV tablo kaydetmek ve PNG görsel üretmek için yardımcı fonksiyonlar tanımlanır.

In [ ]:
def log_output(text=""):
    """
    Print a short notebook message.
    """
    print(str(text))


def log_section(title):
    """
    Print a section header.
    """
    line = "=" * 80

    print("")
    print(line)
    print(str(title))
    print(line)


def log_dataframe(title, df, max_rows=None):
    """
    Print a compact DataFrame preview.
    """
    log_section(title)

    if df is None or len(df) == 0:
        print("[INFO] DataFrame is empty.")
        return

    display_df = df if max_rows is None else df.head(max_rows)
    print(display_df)

    if max_rows is not None and len(df) > max_rows:
        print(f"... total rows: {len(df)}")

def relative_path(path):
    """
    Convert a path to a repository-relative POSIX path when possible.
    """
    path = Path(path)

    try:
        resolved_path = path.resolve()
    except FileNotFoundError:
        resolved_path = path.absolute()

    try:
        return resolved_path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return resolved_path.as_posix()


def check_path_exists(path, expected_type):
    """
    Check whether a path exists with the expected type.
    """
    path = Path(path)

    if expected_type == "directory":
        exists = path.exists() and path.is_dir()
    elif expected_type == "file":
        exists = path.exists() and path.is_file()
    else:
        raise ValueError(f"Unknown expected_type: {expected_type}")

    return {
        "path": relative_path(path),
        "expected_type": expected_type,
        "exists": exists,
        "status": "OK" if exists else "MISSING",
    }


def count_files_by_extension(directory, allowed_extensions):
    """
    Count files with selected extensions in a directory.
    """
    directory = Path(directory)

    if not directory.exists() or not directory.is_dir():
        return 0

    return sum(
        1
        for file_path in directory.iterdir()
        if file_path.is_file() and file_path.suffix.lower() in allowed_extensions
    )


def save_dataframe_csv(df, output_path, overwrite=True, note=""):
    """
    Save a DataFrame as CSV.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        result_df = pd.read_csv(output_path)
        log_output(f"[LOAD] Existing CSV loaded: {relative_path(output_path)}")
    else:
        if output_path.exists() and overwrite:
            log_output(f"[UPDATE] CSV updated: {relative_path(output_path)}")
        else:
            log_output(f"[SAVE] CSV saved: {relative_path(output_path)}")

        df.to_csv(output_path, index=False)
        result_df = df

    return result_df


def save_figure(fig, output_path, overwrite=True, note=""):
    """
    Save a Matplotlib figure as PNG.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        log_output(f"[SKIP] Existing figure kept: {relative_path(output_path)}")
        plt.close(fig)
    else:
        if output_path.exists() and overwrite:
            log_output(f"[UPDATE] Figure updated: {relative_path(output_path)}")
        else:
            log_output(f"[SAVE] Figure saved: {relative_path(output_path)}")

        fig.savefig(output_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    return output_path


def log_figure(title, description, figure_path):
    """
    Print a saved figure reference.
    """
    log_output(f"{title}: {relative_path(figure_path)}")

## 5. Başlangıç Bilgisi

Bu bölümde çalıştırma zamanı ve notebookun kullandığı temel klasör yolları ekrana yazdırılır.

In [ ]:
log_section("00 Project Setup Check Started")
log_output(f"[INFO] Run timestamp: {RUN_TIMESTAMP}")
log_output(f"[INFO] Project root: {PROJECT_ROOT}")
log_output(f"[INFO] Notebook directory: {NOTEBOOK_DIR}")
log_output(f"[INFO] Output directory: {OUTPUT_DIR}")

## 6. Proje Klasör Kontrolü

Bu bölümde repository kökü, ana klasörler, `traditional_ml` yaklaşım klasörü ve bu notebookun input-output klasörleri kontrol edilir.

In [ ]:
project_directory_records = []

project_directory_records.append({
    "check_group": "project_root",
    "item": "PROJECT_ROOT",
    **check_path_exists(PROJECT_ROOT, "directory"),
})

for folder_name in EXPECTED_TOP_LEVEL_FOLDERS:
    project_directory_records.append({
        "check_group": "top_level_folder",
        "item": folder_name,
        **check_path_exists(PROJECT_ROOT / folder_name, "directory"),
    })

for file_name in EXPECTED_REPOSITORY_FILES:
    project_directory_records.append({
        "check_group": "repository_file",
        "item": file_name,
        **check_path_exists(PROJECT_ROOT / file_name, "file"),
    })

project_directory_records.append({
    "check_group": "approach_root",
    "item": APPROACH_NAME,
    **check_path_exists(APPROACH_DIR, "directory"),
})

for folder_name in EXPECTED_APPROACH_FOLDERS:
    project_directory_records.append({
        "check_group": "approach_folder",
        "item": folder_name,
        **check_path_exists(APPROACH_DIR / folder_name, "directory"),
    })

setup_stage_paths = [
    NOTEBOOK_DIR,
    OUTPUT_DIR,
    TABLES_DIR,
    FIGURES_DIR,
]

for path in setup_stage_paths:
    project_directory_records.append({
        "check_group": "setup_stage_path",
        "item": path.relative_to(APPROACH_DIR).as_posix(),
        **check_path_exists(path, "directory"),
    })

project_directory_check_df = pd.DataFrame(project_directory_records)

project_directory_check_df = save_dataframe_csv(
    project_directory_check_df,
    TABLES_DIR / "project_directory_check.csv",
    overwrite=OVERWRITE_TABLES,
    note="Repository and traditional ML setup path check.",
)

log_dataframe("Project Directory Check", project_directory_check_df)

project_directory_check_df

## 7. Ham Veri Klasör Kontrolü

Bu bölümde Dataset1, Dataset2 ve Dataset3 için beklenen ham veri klasörleri ve split alt klasörleri kontrol edilir.

In [ ]:
dataset_folder_records = []

for dataset_name, dataset_info in EXPECTED_DATASETS.items():
    dataset_dir = RAW_DATA_DIR / dataset_name

    dataset_folder_records.append({
        "dataset_name": dataset_name,
        "dataset_role": dataset_info["role"],
        "check_group": "dataset_root",
        "split": None,
        "item": dataset_name,
        **check_path_exists(dataset_dir, "directory"),
    })

    for split in dataset_info["splits"]:
        split_dir = dataset_dir / split

        dataset_folder_records.append({
            "dataset_name": dataset_name,
            "dataset_role": dataset_info["role"],
            "check_group": "split_folder",
            "split": split,
            "item": split,
            **check_path_exists(split_dir, "directory"),
        })

        for subfolder in EXPECTED_SPLIT_SUBFOLDERS:
            subfolder_path = split_dir / subfolder

            dataset_folder_records.append({
                "dataset_name": dataset_name,
                "dataset_role": dataset_info["role"],
                "check_group": "split_subfolder",
                "split": split,
                "item": subfolder,
                **check_path_exists(subfolder_path, "directory"),
            })

    for file_name in EXPECTED_DATASET_LEVEL_FILES:
        file_path = dataset_dir / file_name

        dataset_folder_records.append({
            "dataset_name": dataset_name,
            "dataset_role": dataset_info["role"],
            "check_group": "dataset_level_file",
            "split": None,
            "item": file_name,
            **check_path_exists(file_path, "file"),
        })

dataset_folder_check_df = pd.DataFrame(dataset_folder_records)

dataset_folder_check_df = save_dataframe_csv(
    dataset_folder_check_df,
    TABLES_DIR / "dataset_folder_check.csv",
    overwrite=OVERWRITE_TABLES,
    note="Raw dataset klasör ve dosya kontrolü.",
)

log_dataframe("Raw Dataset Klasör Kontrolü", dataset_folder_check_df)

dataset_folder_check_df

## 8. Ham Veri Dosya Sayımı

Bu bölümde her veri seti ve split için görüntü dosyaları ile etiket dosyaları sayılır.

In [ ]:
file_count_records = []

for dataset_name, dataset_info in EXPECTED_DATASETS.items():
    dataset_dir = RAW_DATA_DIR / dataset_name

    for split in dataset_info["splits"]:
        images_dir = dataset_dir / split / "images"
        labels_dir = dataset_dir / split / "labels"

        image_count = count_files_by_extension(images_dir, IMAGE_EXTENSIONS)
        label_count = count_files_by_extension(labels_dir, LABEL_EXTENSIONS)

        if not images_dir.exists():
            status = "MISSING_IMAGES_DIR"
        elif not labels_dir.exists():
            status = "MISSING_LABELS_DIR"
        elif image_count == 0:
            status = "NO_IMAGES_FOUND"
        elif label_count == 0:
            status = "NO_LABELS_FOUND"
        else:
            status = "OK"

        file_count_records.append({
            "dataset_name": dataset_name,
            "dataset_role": dataset_info["role"],
            "split": split,
            "image_count": image_count,
            "label_count": label_count,
            "image_label_count_difference": image_count - label_count,
            "status": status,
            "images_dir": relative_path(images_dir),
            "labels_dir": relative_path(labels_dir),
        })

raw_dataset_file_count_check_df = pd.DataFrame(file_count_records)

raw_dataset_file_count_check_df = save_dataframe_csv(
    raw_dataset_file_count_check_df,
    TABLES_DIR / "raw_dataset_file_count_check.csv",
    overwrite=OVERWRITE_TABLES,
    note="Raw dataset görüntü ve label dosya sayımı.",
)

log_dataframe("Raw Dataset Dosya Sayımı", raw_dataset_file_count_check_df)

raw_dataset_file_count_check_df

## 9. Kurulum Durum Özeti

Bu bölümde klasör kontrolleri ve veri kontrollerinden genel bir hazırlık durumu çıkarılır.

In [ ]:
project_missing_count = int((project_directory_check_df["status"] == "MISSING").sum())
dataset_missing_count = int((dataset_folder_check_df["status"] == "MISSING").sum())
file_count_problem_count = int((raw_dataset_file_count_check_df["status"] != "OK").sum())

overall_status = "READY" if (
    project_missing_count == 0
    and dataset_missing_count == 0
    and file_count_problem_count == 0
) else "NEEDS_REVIEW"

setup_status_summary_df = pd.DataFrame([
    {
        "run_timestamp": RUN_TIMESTAMP,
        "project_root": relative_path(PROJECT_ROOT),
        "project_directory_missing_count": project_missing_count,
        "dataset_folder_missing_count": dataset_missing_count,
        "file_count_problem_count": file_count_problem_count,
        "overall_status": overall_status,
    }
])

setup_status_summary_df = save_dataframe_csv(
    setup_status_summary_df,
    TABLES_DIR / "setup_status_summary.csv",
    overwrite=OVERWRITE_TABLES,
    note="Setup durum özeti.",
)

log_dataframe("Setup Durum Özeti", setup_status_summary_df)

setup_status_summary_df

## 10. Kurulum Görselleri

Bu bölümde klasör ve veri kontrollerini özetleyen basit görseller oluşturulur.

Görseller bu notebookun `figures` klasörüne kaydedilir.

In [ ]:
figure_info = []

# Şekil 1 — Setup kontrol durumları
status_counts = pd.concat(
    [
        project_directory_check_df[["status"]],
        dataset_folder_check_df[["status"]],
    ],
    ignore_index=True,
)["status"].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
status_counts.plot(kind="bar", ax=ax)

ax.set_title("Setup Kontrol Durumları")
ax.set_xlabel("Durum")
ax.set_ylabel("Sayı")
ax.tick_params(axis="x", rotation=0)

figure_path = save_figure(
    fig,
    FIGURES_DIR / "setup_status_counts.png",
    overwrite=OVERWRITE_FIGURES,
    note="Setup OK/MISSING durum sayıları.",
)

figure_info.append({
    "path": figure_path,
    "title": "Şekil 1 — Setup Kontrol Durumları",
    "description": (
        "Bu grafik, proje klasörü ve raw dataset kontrollerinde kaç öğenin "
        "başarılı geçtiğini veya eksik olduğunu gösterir."
    ),
})


# Şekil 2 — Raw dataset görüntü ve label sayıları
file_counts_plot_df = raw_dataset_file_count_check_df.copy()
file_counts_plot_df["dataset_split"] = (
    file_counts_plot_df["dataset_name"] + " / " + file_counts_plot_df["split"]
)

fig, ax = plt.subplots(figsize=(10, 5))
file_counts_plot_df.set_index("dataset_split")[["image_count", "label_count"]].plot(
    kind="bar",
    ax=ax,
)

ax.set_title("Raw Dataset Görüntü ve Label Sayıları")
ax.set_xlabel("Dataset / Split")
ax.set_ylabel("Dosya Sayısı")
ax.tick_params(axis="x", rotation=45)

figure_path = save_figure(
    fig,
    FIGURES_DIR / "raw_dataset_file_counts.png",
    overwrite=OVERWRITE_FIGURES,
    note="Dataset split bazlı görüntü ve label sayıları.",
)

figure_info.append({
    "path": figure_path,
    "title": "Şekil 2 — Raw Dataset Görüntü ve Label Sayıları",
    "description": (
        "Bu grafik, Dataset1, Dataset2 ve Dataset3 için train, valid ve test "
        "splitlerindeki görüntü ve label dosya sayılarını karşılaştırır."
    ),
})


# Şekil 3 — Setup problem özeti
summary_counts = pd.Series({
    "project_missing": project_missing_count,
    "dataset_missing": dataset_missing_count,
    "file_count_problem": file_count_problem_count,
})

fig, ax = plt.subplots(figsize=(7, 4))
summary_counts.plot(kind="bar", ax=ax)

ax.set_title("Setup Problem Özeti")
ax.set_xlabel("Problem Türü")
ax.set_ylabel("Sayı")
ax.tick_params(axis="x", rotation=30)

figure_path = save_figure(
    fig,
    FIGURES_DIR / "missing_items_summary.png",
    overwrite=OVERWRITE_FIGURES,
    note="Setup eksik/problemli öğe özeti.",
)

figure_info.append({
    "path": figure_path,
    "title": "Şekil 3 — Setup Problem Özeti",
    "description": (
        "Bu grafik, proje klasörü eksikleri, dataset klasörü eksikleri ve dosya sayımı "
        "problemlerini özetler."
    ),
})

log_section("Oluşturulan Görseller")

figure_table_df = pd.DataFrame([
    {
        "figure_title": item["title"],
        "figure_path": relative_path(item["path"]),
        "description": item["description"],
    }
    for item in figure_info
])

log_dataframe("Oluşturulan Görsel Listesi", figure_table_df)

for item in figure_info:
    log_figure(item["title"], item["description"], item["path"])

figure_info

## 11. Final Durum

Bu bölümde genel kurulum durumu ve üretilen tablo-görsel klasörleri ekrana yazdırılır.

In [ ]:
log_section("00 Project Setup Check Completed")
log_output(f"Overall setup status: {overall_status}")
log_output(f"Project root: {relative_path(PROJECT_ROOT)}")
log_output(f"Output directory: {relative_path(OUTPUT_DIR)}")
log_output(f"Tables directory: {relative_path(TABLES_DIR)}")
log_output(f"Figures directory: {relative_path(FIGURES_DIR)}")

if overall_status == "READY":
    log_output("[NEXT] Continue with 01_exploration.")
else:
    log_output("[REVIEW] Check the setup output tables before continuing.")